In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("10-diamonds.csv")

In [4]:
df.head()

,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [5]:
df = df.drop(["Unnamed: 0"], axis=1)

In [6]:
# burda yukarıdaki kodda x, y, z değerleri 0 olan satırları düşürüyoruz çünkü bu değerler hatalıdır.
#  Bu değerler 0 olamaz. Bu yüzden bu satırları düşürüyoruz.
df = df.drop(df[df["x"]==0].index)
df = df.drop(df[df["y"]==0].index)
df = df.drop(df[df["z"]==0].index)

In [7]:
#burda da 75 den büyük ve 45 den küçük olan depth değerlerini düşürüyoruz. Çünkü bu değerler hatalıdır. 
# outlier olarak kabul edebiliriz. Bu yüzden bu satırları düşürüyoruz.
df = df[(df["depth"]<75)&(df["depth"]>45)]
df = df[(df["table"]<80)&(df["table"]>40)]
df = df[(df["y"]<30)]
df = df[(df["z"]<30)&(df["z"]>2)]

In [8]:

X= df.drop(["price"],axis =1)
y= df["price"]

In [9]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [10]:
from sklearn.model_selection import train_test_split


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.25, random_state=15)


In [12]:
from sklearn.preprocessing import LabelEncoder


In [13]:

## Daha önce sütunları etiket kodlayıcı ile şu şekilde kodlamıştık ve çalışmıştı.
# Ancak kodlayıcıları kaydetmek istiyorsak, onları ayırmamız gerekirdi.
# Bu yüzden yeni bir kodlama kullanacağım.labelencoder değil de encoder kullanacağım.
#  Çünkü encoder ile kodladığımızda, kodlayıcıyı kaydedebiliriz ve daha sonra tekrar kullanabiliriz.
#label_encoder = LabelEncoder()
#for col in ['cut', 'color', 'clarity']:
#    X_train[col] = label_encoder.fit_transform(X_train[col])
#    X_test[col] = label_encoder.transform(X_test[col])

In [14]:
#encoders değişkeni ile her bir sütun için ayrı bir encoder oluşturuyoruz ve bu encoderları kaydedebiliriz.
#  bu durum labelencoder ile mümkün değildir. çünkü labelencoder tek bir encoder ile tüm sütunları kodlar ve bu yüzden kaydedilemez. 
# o yüzden her bir sütun için ayrı bir encoder oluşturuyoruz ve bu encoderları kaydedebiliriz.
encoders = {}
for col in ['cut', 'color', 'clarity']:
    encoders[col] = LabelEncoder()
    X_train[col] = encoders[col].fit_transform(X_train[col])
    X_test[col] = encoders[col].transform(X_test[col])


In [15]:

X_train.head()

,carat,cut,color,clarity,depth,table,x,y,z
15200,1.15,2,4,2,62.4,54.0,6.71,6.76,4.20
14632,1.11,3,1,2,61.3,58.0,6.66,6.61,4.07
19151,1.21,1,2,5,63.7,58.0,6.67,6.71,4.26
29299,0.30,2,5,5,61.5,58.0,4.28,4.31,2.64
9983,1.00,4,2,2,63.1,57.0,6.37,6.33,4.01


In [16]:
from sklearn.preprocessing import StandardScaler


In [17]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [18]:
from sklearn.svm import SVR
svr=SVR(C=1000, gamma=0.1, kernel='rbf')
# Bunu daha önce yaptığımız hiperparametre ayarlamalarından elde ediyoruz.
# # refer: https://github.com/atilsamancioglu/MachineLearningNotebooks/blob/main/10-SVMRegressor.ipynb
#burda ne yapıyoruz? svr modelini oluşturuyoruz ve parametrelerini belirliyoruz.
#  C=1000, gamma=0.1, kernel='rbf' olarak belirliyoruz. 
# Bu parametreler daha önce yaptığımız hiperparametre optimizasyonundan elde ettiğimiz en iyi parametrelerdir.

In [19]:
from sklearn.metrics import r2_score


In [20]:
svr.fit(X_train_scaled, y_train)
y_pred=svr.predict(X_test_scaled)
score=r2_score(y_test,y_pred)
print("R2 Score", score)
#burda ne yapıyoruz? svr modelini eğitiyoruz ve test verisi ile tahmin yapıyoruz. Daha sonra r2 skoru hesaplıyoruz ve ekrana yazdırıyoruz.

R2 Score 0.9452198447140456


In [21]:
encoders

{'cut': LabelEncoder(), 'color': LabelEncoder(), 'clarity': LabelEncoder()}

In [22]:
scaler

StandardScaler()

In [23]:
svr

SVR(C=1000, gamma=0.1)

In [24]:
import pickle



In [25]:
# pickle.dump  burdaki değerleri alıp dosyaya kaydeder
# pickle.load  yüklemek


In [26]:
# Eğitilmiş modeli ve veri ön işleme araçlarını daha sonra kullanmak üzere tek bir pakette kaydediyoruz.
# 'wb' (write binary) modu ile dosyayı ikili yazma modunda açarız.
# 'with' bloğu, işlem tamamlandığında dosyanın otomatik kapanmasını sağlar.
with open('30-diamond_model_complete.pkl', 'wb') as f:
    
    # Tüm gerekli nesneleri bir sözlük (dictionary) içinde gruplayarak 'pickle' ile dosyaya yazdırıyoruz.
    pickle.dump({
        'model': svr,          # Tahmini yapacak olan asıl makine öğrenmesi modelimiz (Destek Vektör Regresyonu)
        'encoders': encoders,  # Yeni gelecek kategorik verileri, modelin anladığı sayılara dönüştürecek araçlar
        'scaler': scaler       # Yeni gelecek sayısal verileri, eğitimdeki ile aynı ölçeğe getirecek araç
    }, f)

In [27]:
X_test_scaled

array([[ 0.6196541 , -0.53873069,  0.82581707, ...,  0.83254363,
         0.78315004,  0.76989728],
       [ 1.52658497, -0.53873069,  0.23842797, ...,  1.47548072,
         1.50279799,  1.44915416],
       [-0.56146517, -0.53873069,  0.23842797, ..., -0.45333055,
        -0.42226029, -0.47299831],
       ...,
       [-0.2029111 ,  0.43597985, -0.93635025, ..., -0.02470582,
        -0.08042751, -0.01052553],
       [ 1.5687678 ,  0.43597985,  1.41320618, ...,  1.51119945,
         1.55677159,  1.43470189],
       [-0.94111064, -0.53873069,  0.23842797, ..., -1.02483018,
        -1.06994345, -1.0944461 ]])

In [28]:
#x_test_scaled'i kaydedip PyCharm'da çalışıp çalışmadığını bakıcaz

In [29]:
pd.DataFrame(X_test_scaled).to_csv("30_testdatascaled.csv", index=False)
